# ParsingAI — Colab 版
依序執行每個 Cell。

In [ ]:
# ── Cell 1: 安裝套件 & 設定 ────────────────────────────────────────────────────
!pip install ollama -q
!apt-get install -y zstd pciutils -q
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

MODEL = 'qwen3:4b'
print(f'拉取模型：{MODEL}')
!ollama pull {MODEL}
print('✓ 完成')

In [ ]:
# ── Cell 2: 上傳處理過的 log txt ───────────────────────────────────────────────
import re
from google.colab import files
from pathlib import Path

uploaded = files.upload()
log_filename = list(uploaded.keys())[0]
cleaned_log = Path(log_filename).read_text(encoding='utf-8', errors='ignore')

# 從 log 自動抓 project / fw_version
fw_match = re.search(r'Firmware Version\s*:\s*(\S+)', cleaned_log)
fw_version = fw_match.group(1) if fw_match else ''

if 'LJ1(SSSTC)' in cleaned_log or re.search(r'FG\d+', cleaned_log):
    project = 'PJ1'
elif 'SSSTC ER3' in cleaned_log or 'FWInfo' in cleaned_log:
    project = 'ER3'
else:
    project = 'PJ1'  # 預設

est_tokens = max(1, len(cleaned_log) // 4)
print(f'檔案      : {log_filename}')
print(f'Project   : {project}')
print(f'FW Version: {fw_version}')
print(f'預估 token: {est_tokens:,}')

In [ ]:
# ── Cell 3: 執行 LLM 分析 ──────────────────────────────────────────────────────
# 只需修改這行
issue = '請輸入問題描述'

import subprocess, time, httpx
try:
    httpx.get('http://127.0.0.1:11434', timeout=2)
except Exception:
    print('啟動 Ollama server...')
    subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)

from IPython.display import display, clear_output
import ipywidgets as widgets
import ollama

SYSTEM_PROMPT = """你是一位 SSD 韌體工程師，專門分析 SSD 故障 log。
請根據提供的 log 資料，分析問題根因並提供具體建議。
回答請使用繁體中文。"""

USER_PROMPT = f"""型號：{project}
韌體版本：{fw_version}
問題描述：{issue}

以下是處理過的 log 資料：
{cleaned_log}
"""

llm_out = widgets.Output()
display(llm_out)
full_text = ''

for chunk in ollama.chat(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': USER_PROMPT},
    ],
    options={'num_ctx': 131072},
    stream=True,
):
    token = chunk.get('message', {}).get('content', '')
    full_text += token
    with llm_out:
        clear_output(wait=True)
        print(full_text, end='', flush=True)